<a href="https://colab.research.google.com/github/KiaNoForte/Cryptography/blob/main/CryptoLab03_Cryptographic_Security.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CYBR 3570 Crypto Lab 03
## Cryptographic Security: Quantifying “Impossible”

**Big question:** When we say a cryptographic system is secure, what claim are we actually making?

This notebook turns the reading into executable experiments. You will model attacker effort, compute security levels in bits, compare key sizes, and connect cryptographic security claims to real engineering decisions.

> Course theme: cryptography is not magic. It is a set of carefully scoped security claims under explicit assumptions.

## Learning Objectives

By the end of this notebook, you should be able to:

1. Distinguish unconditional security from computational security.
2. Explain what a `(t, ε)` security claim means.
3. Convert attack cost into a bit-security estimate using `log2`.
4. Estimate brute-force success probability for a symmetric key.
5. Explain how parallelism, memory, precomputation, and multiple targets change attack cost.
6. Generate symmetric keys safely using Python's `secrets` module.
7. Explain why key size, implementation quality, and system design all matter.

## Reading Check

Answer these in Markdown before running the code.

1. What is the difference between unconditional security and computational security?

Unconditional security is a system where it cannot be broken even if the adversary has infinite
2. Why does a one-time pad provide unconditional security only under strict key requirements?
3. What does “128-bit security” mean in practical terms?
4. Why is key size only an upper bound on security level?
5. Why can a provably secure scheme still fail in a real implementation?

## Part 1: Security as an Attacker-Cost Claim

In everyday language, we often say a system is either “secure” or “insecure.” In cryptography, that is not precise enough.

A more useful claim is:

> An attacker with some bounded amount of computation has at most some bounded probability of success.

This is often written as `(t, ε)` security, where:

- `t` is the number of operations the attacker is allowed to perform.
- `ε` is the maximum probability that the attacker succeeds.

For a symmetric cipher with an ideal `n`-bit key, brute force succeeds with probability approximately:

\[
\epsilon = \frac{t}{2^n}
\]

as long as `t <= 2^n`.

In [3]:
import math
import secrets
from pathlib import Path

import matplotlib.pyplot as plt

SECONDS_PER_YEAR = 60 * 60 * 24 * 365.25

def brute_force_success_probability(key_bits: int, attempts: int) -> float:
    """Return the probability of success after a given number of key attempts."""
    if key_bits <= 0:
        raise ValueError("key_bits must be positive")
    if attempts < 0:
        raise ValueError("attempts cannot be negative")
    key_space = 2 ** key_bits
    return min(attempts / key_space, 1.0)

for attempts_power in [1, 32, 64, 96, 128]:
    attempts = 2 ** attempts_power
    probability = brute_force_success_probability(128, attempts)
    print(f"Trying 2^{attempts_power:>3} keys against a 128-bit key: probability = {probability:.3e}")

Trying 2^  1 keys against a 128-bit key: probability = 5.877e-39
Trying 2^ 32 keys against a 128-bit key: probability = 1.262e-29
Trying 2^ 64 keys against a 128-bit key: probability = 5.421e-20
Trying 2^ 96 keys against a 128-bit key: probability = 2.328e-10
Trying 2^128 keys against a 128-bit key: probability = 1.000e+00


### Reflection

Why is trying `2^64` keys against a 128-bit key still not a realistic break, even though `2^64` sounds enormous?

2^64 is a massive number in absolute terms yes, it represents only an infinitesimal fraction of a 128-bit key space.Trying 2^64 keys against a 128-bit key leaves the attacker $18.4$ quintillion times farther away from searching the full key space than a 64-bit key would require.

## Part 2: Measuring Security in Bits

If an attack requires approximately `N` operations, its bit security is:

\[
\log_2(N)
\]

For example, one million operations is roughly `2^20`, so an attack requiring one million operations has only about 20 bits of security.

In [4]:
def bit_security(operations: int | float) -> float:
    """Convert an attack cost in operations to a bit-security estimate."""
    if operations <= 0:
        raise ValueError("operations must be positive")
    return math.log2(operations)

examples = [1_000, 1_000_000, 2**40, 2**64, 2**80, 2**128]
for ops in examples:
    print(f"{ops:>40,} operations ≈ {bit_security(ops):6.2f} bits of security")

                                   1,000 operations ≈   9.97 bits of security
                               1,000,000 operations ≈  19.93 bits of security
                       1,099,511,627,776 operations ≈  40.00 bits of security
              18,446,744,073,709,551,616 operations ≈  64.00 bits of security
       1,208,925,819,614,629,174,706,176 operations ≈  80.00 bits of security
340,282,366,920,938,463,463,374,607,431,768,211,456 operations ≈ 128.00 bits of security


### Exercise

Compute the approximate bit security of each attack cost:

1. `10^9` operations
2. `10^12` operations
3. `10^18` operations
4. `2^112` operations

Then answer: which of these would you consider realistic for a determined attacker today? Explain your assumption about the attacker.


10^18 would be highly realistic for a determined attacker. Standard GPU clusters routinely achieve trillions of operations per second which can mae this doable in days or weeks. The attacker would sufficent resources since they would have high-end hardware, parallel software and energy budget spanning to thousands of dollars if not more and bounded limits included.

In [6]:
# Try your own calculations here.
attack_costs = [10**9, 10**12, 10**18, 2**112]
for cost in attack_costs:
    print(f"{cost:.3e} operations ≈ {bit_security(cost):.2f} bits")

1.000e+09 operations ≈ 29.90 bits
1.000e+12 operations ≈ 39.86 bits
1.000e+18 operations ≈ 59.79 bits
5.192e+33 operations ≈ 112.00 bits


## Part 3: Estimating Brute-Force Time

Bit security is a useful abstraction, but attackers care about time, money, hardware, energy, and targets.

This function estimates brute-force time under a simplified model:

- The key has `key_bits` bits.
- Each core tests `keys_per_second` keys.
- There are `cores` independent workers.
- The attacker only needs to break one of `targets` possible keys.

This model is intentionally simplified, but it helps reason quantitatively.

In [ ]:
def brute_force_years(
    key_bits: int,
    keys_per_second: float,
    cores: int = 1,
    targets: int = 1,
    average_case: bool = True,
) -> float:
    """Estimate brute-force search time in years."""
    if key_bits <= 0:
        raise ValueError("key_bits must be positive")
    if keys_per_second <= 0:
        raise ValueError("keys_per_second must be positive")
    if cores <= 0 or targets <= 0:
        raise ValueError("cores and targets must be positive")
    attempts = 2 ** key_bits
    if average_case:
        attempts /= 2
    attempts /= targets
    attempts /= cores
    seconds = attempts / keys_per_second
    return seconds / SECONDS_PER_YEAR

for bits in [40, 56, 64, 80, 128, 256]:
    years = brute_force_years(bits, keys_per_second=1e12, cores=1)
    print(f"{bits:>3}-bit key at 1 trillion keys/sec: {years:.3e} years")

### Interpretation

At one trillion key tests per second, short key sizes collapse quickly. The lesson is not that every attacker has this exact hardware. The lesson is that exponential growth dominates.

A few dozen bits can be the difference between “homework exercise” and “not happening in the lifetime of the universe.”

In [ ]:
key_sizes = [40, 56, 64, 80, 96, 112, 128]
years = [brute_force_years(bits, keys_per_second=1e12) for bits in key_sizes]

plt.figure(figsize=(8, 4.5))
plt.semilogy(key_sizes, years, marker='o')
plt.xlabel('Key size in bits')
plt.ylabel('Estimated average brute-force time (years, log scale)')
plt.title('Brute-force time grows exponentially')
plt.grid(True, which='both')
plt.show()

## Part 4: Parallelism and Multiple Targets

Some attacks can be split across many machines. Brute-force key search is one of them.

If an attacker has `2^20` cores and is happy to break any one of `2^20` targets, the effective brute-force cost is reduced by `2^40`.

That is substantial, but it does **not** make 128-bit security weak.

In [ ]:
base = brute_force_years(128, keys_per_second=1e9, cores=1, targets=1)
reduced = brute_force_years(128, keys_per_second=1e9, cores=2**20, targets=2**20)
print(f"Single core, one target:       {base:.3e} years")
print(f"2^20 cores, 2^20 targets:     {reduced:.3e} years")
print(f"Reduction factor:             {base / reduced:.3e}")

### Discussion

1. Why does the number of targets matter in large-scale attacks?

An adversary does not need to compromise a specific victim's key. Breaking an single key among millions of valid targets would be a successful breach nonetheless.

2. Why does this not mean that 128-bit security is automatically unsafe?

When accounting for millions to billions of potential targets, 128-bit security provides a large margin of protection

3. What assumptions in this model are unrealistic or incomplete?

Well you have linear scaling with no overhead, there's infinite energy or funding which ignores the physical limits of thermodynamics and then target detection costs which the model assumes verifying whether a trial key will decrypt data successfully takes the atomic step. Then the protocol context which assumes the static and pure brute-force setting.

## Part 5: Key Size vs Security Level

A key's length gives an upper bound on the security level, but it does not guarantee that level.

A 128-bit symmetric key can provide 128-bit security **only if** the best attack is brute force.

Security can be lower when:

- the cipher has a shortcut attack,
- randomness is weak,
- keys are reused incorrectly,
- implementation leaks timing or power information,
- the system stores the key insecurely,
- a compatibility mode allows weak parameters.

### Scenario Analysis

For each system below, estimate whether the claimed security level is believable.

| Scenario | Claimed Security | Your Evaluation |
|---|---:|---|
| AES-128 with keys generated using `secrets.token_bytes(16)` | 128-bit | |
| AES-256 where the key is committed to GitHub | 256-bit | |
| A custom cipher with a 256-bit key and no public cryptanalysis | 256-bit | |
| A password-derived key from the password `password123` | depends | |
| RSA with an outdated 512-bit modulus | high? | |

Write your answers in Markdown below.

* **AES-128 with keys generated using `secrets.token_bytes(16)`:** **Believable (128-bit).** `secrets` provides cryptographic-quality randomness with full 128-bit entropy, matching the cipher's maximum theoretical strength.
* **AES-256 where the key is committed to GitHub:** **Unbelievable (0-bit).** Publicly exposing the private key eliminates all security regardless of algorithm length.
* **A custom cipher with a 256-bit key and no public cryptanalysis:** **Unbelievable (Unknown/Low).** Unvetted "roll-your-own" cryptography almost always contains structural flaws that dramatically reduce real-world bit security well below key size.
* **A password-derived key from `password123`:** **Unbelievable (Low).** The underlying secret lacks sufficient entropy; simple dictionary lookups or targeted wordlists break this almost instantly.
* **RSA with an outdated 512-bit modulus:** **Unbelievable (Insecure).** 512-bit RSA key moduli are easily factorable with modern compute power and offer less than 60 bits of security.

## Part 6: Generating Symmetric Keys Safely

For symmetric keys, key generation is conceptually simple: request enough random bytes from a cryptographic PRNG.

In Python, use `secrets` or a trusted cryptographic library. Do **not** use the `random` module for cryptographic keys.

In [ ]:
def generate_symmetric_key(bits: int = 128) -> bytes:
    """Generate a symmetric key using Python's cryptographic randomness source."""
    if bits <= 0 or bits % 8 != 0:
        raise ValueError("bits must be a positive multiple of 8")
    return secrets.token_bytes(bits // 8)

key_128 = generate_symmetric_key(128)
key_256 = generate_symmetric_key(256)
print(f"128-bit key: {key_128.hex()}")
print(f"256-bit key: {key_256.hex()}")
print(len(key_128), 'bytes')
print(len(key_256), 'bytes')

### Check Yourself

Why is `secrets.token_bytes(16)` appropriate for a 128-bit symmetric key, while `random.getrandbits(128)` is not appropriate for cryptographic key generation?

This oeprations pulls randomness from the operating system's Cryptographically Secure Psuedorandom Number Generator. This'll rely on hardware enviormental noise making the output non-deterministic and cryptographically unpredictable. In contrast, random.getrandbits(128) uses the Mersenne Twister (MT19937) algorithm. Mersenne Twister is designed purely for statistical randomness, not security; its internal state can be completely reconstructed after observing just 624 32-bit outputs, allowing an attacker to predict future keys.

## Part 7: Toolkit Integration

This week adds a small `security` module to your cryptographic toolkit. The module is not an implementation of a cipher. Instead, it contains helper functions for reasoning about security levels and brute-force cost.

Suggested path:

```text
crypto_toolkit/security/security_levels.py
```

Run the next cell from your toolkit repository root to create the file.

In [7]:
module_path = Path('crypto_toolkit/security')
module_path.mkdir(parents=True, exist_ok=True)
(module_path / '__init__.py').write_text('from .security_levels import bit_security, brute_force_success_probability, brute_force_years, generate_symmetric_key\n')

module_code = '''
from __future__ import annotations

import math
import secrets

SECONDS_PER_YEAR = 60 * 60 * 24 * 365.25


def bit_security(operations: int | float) -> float:
    """Convert an attack cost in operations to a bit-security estimate."""
    if operations <= 0:
        raise ValueError('operations must be positive')
    return math.log2(operations)


def brute_force_success_probability(key_bits: int, attempts: int) -> float:
    """Return the probability of brute-force success after a number of attempts."""
    if key_bits <= 0:
        raise ValueError('key_bits must be positive')
    if attempts < 0:
        raise ValueError('attempts cannot be negative')
    return min(attempts / (2 ** key_bits), 1.0)


def brute_force_years(key_bits: int, keys_per_second: float, cores: int = 1, targets: int = 1, average_case: bool = True) -> float:
    """Estimate brute-force search time in years under a simplified model."""
    if key_bits <= 0:
        raise ValueError('key_bits must be positive')
    if keys_per_second <= 0:
        raise ValueError('keys_per_second must be positive')
    if cores <= 0 or targets <= 0:
        raise ValueError('cores and targets must be positive')
    attempts = 2 ** key_bits
    if average_case:
        attempts /= 2
    attempts /= cores
    attempts /= targets
    seconds = attempts / keys_per_second
    return seconds / SECONDS_PER_YEAR


def generate_symmetric_key(bits: int = 128) -> bytes:
    """Generate a symmetric key using Python's cryptographic randomness source."""
    if bits <= 0 or bits % 8 != 0:
        raise ValueError('bits must be a positive multiple of 8')
    return secrets.token_bytes(bits // 8)
'''

(module_path / 'security_levels.py').write_text(module_code)
print('Created crypto_toolkit/security/security_levels.py')

Created crypto_toolkit/security/security_levels.py


In [8]:
from crypto_toolkit.security import (
    bit_security,
    brute_force_success_probability,
    brute_force_years,
    generate_symmetric_key,
)

assert round(bit_security(2**20), 2) == 20.00
assert brute_force_success_probability(8, 256) == 1.0
assert len(generate_symmetric_key(128)) == 16
print('Basic toolkit checks passed.')

Basic toolkit checks passed.


## Part 8: Security Engineering Questions

Answer these in Markdown.

1. A system advertises “AES-256 encryption.” What additional questions would you ask before trusting the claim?

* **Mode of Operation:** Which block cipher mode is used (e.g., GCM, CBC, ECB)? Modes like ECB leave patterns visible in ciphertext, and unauthenticated modes like CBC lack integrity checks.
   * **Key Generation & Management:** How was the key generated (is it from a cryptographically secure PRNG?), and how is it derived, stored, and rotated?
   * **Implementation Quality:** Is the implementation resistant to side-channel attacks (e.g., cache timing or power analysis leaks)?

2. Why might a short-lived 64-bit key be acceptable in a tightly controlled system, while a 64-bit long-term key would be unacceptable?
* **Short-Lived Key:** A 64-bit key provides around $2^{64}$ search space operations. If the key expires in seconds or minutes, an attacker lacks the time and computational budget to complete a brute-force search within its valid lifespan, making the window of vulnerability extremely narrow.
   * **Long-Term Key:** Over months or years, an attacker can harvest ciphertext and easily perform $2^{64}$ operations (or fewer with parallel target attacks) to recover the key and decrypt all historical data associated with it.


3. What is the difference between saying “no one has broken this yet” and saying “this has a security proof”? Which one applies to AES?

* **No One Has Broken This Yet:** Represents empirical security. The algorithm has withstood years of public scrutiny and cryptanalysis by experts, but there is no formal mathematical proof guaranteeing an attack does not exist.
   * **Security Proof:** Represents provable security. The scheme's security is mathematically reduced to a known hard problem (this involves the hardness of factoring large integers or Discrete Logarithm Problems).
   * **Application to AES:** AES relies on **"no one has broken this yet."** Its strength is empirical, backed by decades of intensive cryptanalysis without a practical structural break

4. Explain why a private key stored in source code can destroy the security of an otherwise strong cryptographic system.

* Hardcoding a key in source code moves it into version control (e.g., Git history), build artifacts, and client-side binaries where it can be extracted via reverse engineering or repository leaks.
   * Cryptographic strength relies entirely on Kerckhoffs's principle—that the system remains secure even if everything except the key is public. Exposing the key collapses the entire mathematical boundary of the cipher to zero effort for an attacker, rendering strong primitives useless

5. How does this chapter reinforce the course theme “security is a system property”?

* A strong cryptographic algorithm (like AES-256) is only one primitive in a larger pipeline.
   * System security fails at its weakest point. If key generation, key storage, protocol design, memory management, or implementation side-channels are flawed, the end-to-end security collapses regardless of how mathematically sound the underlying cipher is. Security depends on how all components integrate and function together in practice.

## Final Reflection

Return to the big question:

> When we say a cryptographic system is secure, what claim are we actually making?

Write a 5–7 sentence answer. Your answer should mention attacker resources, probability of success, key generation, and implementation assumptions.


When we state that the crytographic system is safe and secure we are not making a claim of absolute. See this is labeled as unbreakable security vs a computational claim which is bounded by specific parameters. We are asserting that an adversary with a bounded amount of attacker resources which is the following: Time, hardware, memory, and energy. This guarantee relies on the assumption that keys are generated using cryptographically secure key generation processes to prevent prediction or brute-force shortcuts. Furthermore, mathematical primitives are sound and that all underlying implementation assumptions hold true in practice. That means side channels, timing leaks, key storage flaws, or improper protocols cannot be exploited to bypass the algorithm's mathematical strength.

## Submission Checklist

Before submitting:

- [ ] I answered the reading check questions.
- [ ] I ran all code cells.
- [ ] I completed the scenario analysis.
- [ ] I created or updated the `crypto_toolkit/security/security_levels.py` module.
- [ ] I answered the security engineering questions.
- [ ] I completed the final reflection.
- [ ] I committed and pushed my work to GitHub.